# Week6 복습과제 – YOLOv8 모델 실습 (교재 9장 p.591~600)

- 논문: You Only Look Once: Unified, Real-Time Object Detection
- 교재: (파이토치 트랜스포머를 활용한) 자연어 처리와 컴퓨터비전 심층학습
- 실행 환경: Google Colab (런타임 → 런타임 유형 변경 → **GPU**)

## 1. 라이브러리 설치
YOLOv8은 Ultralytics 라이브러리로 제공된다.

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

## 2. 사전학습 모델 불러오기
- `yolov8n.pt`(nano) ~ `yolov8x.pt`(extra large) 중 선택할 수 있다.
- COCO 데이터셋(80개 클래스)으로 사전학습된 가중치를 사용한다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")
print(model.names)          # 클래스 ID → 이름
print(model.info())         # 레이어 수, 파라미터 수, GFLOPs

## 3. 이미지 추론
YOLO v1과 달리 YOLOv8은 **anchor-free** 방식이며, backbone–neck(PAN-FPN)–decoupled head 구조로 3개의 스케일에서 박스를 예측한다.

In [ ]:
results = model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    conf=0.25,      # confidence threshold
    iou=0.7,        # NMS IoU threshold
    save=True,
)

for r in results:
    boxes = r.boxes
    print("박스 개수:", len(boxes))
    print("xyxy :", boxes.xyxy)
    print("conf :", boxes.conf)
    print("class:", [model.names[int(c)] for c in boxes.cls])

## 4. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

plotted = results[0].plot()          # BGR numpy array
plt.figure(figsize=(8, 10))
plt.imshow(plotted[:, :, ::-1])      # BGR → RGB
plt.axis("off")
plt.show()

## 5. 데이터셋으로 미세 조정(Fine-tuning)
- `coco128.yaml`: COCO train2017 앞 128장으로 구성된 예제 데이터셋 (자동 다운로드)
- YAML 파일에는 `path`, `train`, `val`, `names`가 정의되어 있다.

In [ ]:
model = YOLO("yolov8m.pt")

train_results = model.train(
    data="coco128.yaml",
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,          # GPU
    project="runs",
    name="yolov8_coco128",
)

## 6. 학습 결과 확인
학습 과정의 loss(box / cls / dfl)와 mAP 그래프가 `results.png`에 저장된다.

In [ ]:
from IPython.display import Image, display

save_dir = train_results.save_dir
display(Image(filename=f"{save_dir}/results.png", width=900))
display(Image(filename=f"{save_dir}/confusion_matrix.png", width=700))

## 7. 검증(Validation)

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)
print("mAP75   :", metrics.box.map75)

## 8. 학습된 모델로 추론

In [ ]:
best = YOLO(f"{save_dir}/weights/best.pt")
res = best.predict("https://ultralytics.com/images/zidane.jpg", conf=0.25)

plt.figure(figsize=(10, 6))
plt.imshow(res[0].plot()[:, :, ::-1])
plt.axis("off")
plt.show()

## 9. 정리 – YOLO v1 vs YOLOv8
| | YOLO v1 | YOLOv8 |
|---|---|---|
| 예측 방식 | S×S 그리드, 셀당 B개 box | anchor-free, 픽셀(point) 단위 예측 |
| 스케일 | 단일 7×7 | 3개 스케일 (P3, P4, P5) |
| Head | FC 기반 | Decoupled head (분류/회귀 분리) |
| Loss | SSE | BCE(cls) + CIoU + DFL(box) |
| 작은 물체 | 약함 | multi-scale feature로 개선 |